In [1]:
# Importa tudo

from selenium import webdriver
from selenium.webdriver.support.select import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

#Bibliotecas de Sistema
import time
import re
import csv
import os
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd
from contextlib import closing

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

In [2]:
def lista_tipos_pedidos():
    conn = sqlite3.connect('movimentos.db')
    with closing(conn.cursor()) as cursor:
        cursor.execute("SELECT id, resumo FROM pedidos")
        pedidos_lista = [f"ID: {row[0]}. RESUMO : {row[1]}" for row in cursor.fetchall() if row[1]]
    conn.close()
    tipos_pedidos = "\n".join(pedidos_lista)
    return tipos_pedidos

In [3]:
tipos_pedidos = lista_tipos_pedidos()
tipos_pedidos

'ID: 1. RESUMO : Pedido de desistência do processo\nID: 2. RESUMO : Informação de que a parte está ciente.\nID: 3. RESUMO : Solicita a inclusão da parte executada em cadastros de inadimplentes para fins de cobrança de dívida tributária.'

In [4]:
def inserir_pedido(pedido):
    conn = sqlite3.connect('movimentos.db')
    with closing(conn.cursor()) as cursor:
        cursor.execute(
            "INSERT INTO pedidos (resumo) VALUES (?)",
            (pedido,)
        )
        conn.commit()
    conn.close()

In [10]:
pedido = "Solicita a inclusão da parte executada em cadastros de inadimplentes para fins de cobrança de dívida tributária."
inserir_pedido(pedido)

In [11]:
conn = sqlite3.connect('movimentos.db')
with closing(conn.cursor()) as cursor:
    cursor.execute("SELECT id, documentos FROM movimentos WHERE documentos LIKE '%*PET*%' AND resumo IS NULL")
    documentos_pet = cursor.fetchall()
conn.close()
print(documentos_pet)

[]


In [9]:
def ollama_resumo(pedido):
    pergunta_gemma = "Considere o seguinte pedido, identificado por *PET* (os outros pedaços do texto são apenas anexos)" \
    f"{pedido}" \
    "Resuma, da maneira mais objetiva possível, o pedido. Não mencione dados pessoais, como nomes, números de documento, números de processo, valores, etc. O resumo deve ser genérico e breve (uma frase apenas)." \

    resumo = ollama.chat(
        model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
        messages=[{'role': 'user', 'content': f'{pergunta_gemma}'}],    
    )

    return(resumo['message']['content'])


In [12]:
for peticao in documentos_pet:
    pet_id, documentos = peticao
    resumo = ollama_resumo(documentos)
    print(f"ID: {pet_id}, Resumo: {resumo}")
    conn = sqlite3.connect('movimentos.db')
    with closing(conn.cursor()) as cursor:
        cursor.execute(
            "UPDATE movimentos SET resumo = ? WHERE id = ?",
            (resumo, pet_id)
        )
        conn.commit()
    conn.close()